# CuTeDSL 01: CuTeDSL 基础教程

- **Hello World**：如何编写在 GPU 上运行的简单程序
- **数据类型**：CuTe DSL 中的原始数值类型（Int8/16/32/64, Float16/32/64 等）
- **Tensor**：CuTe 的核心数据结构，包括 Engine + Layout
- **TensorSSA**：静态单赋值形式的 tensor 值，用于寄存器级操作
- **打印调试**：在 CuTe 中打印静态和动态值的不同方式
- **CUDA graph**：通过 PyTorch 与 CuTe DSL 使用 CUDA graph

## 一、Hello World



在本节中，我们将使用 CuTe DSL 编写一个在 GPU 上运行的简单 "Hello World" 程序。

- 如何编写同时在 CPU（host）和 GPU（device）上运行的代码
- 如何启动 GPU kernel（在 GPU 上运行的函数）
- 基本的 CUDA 概念，如 thread 和 block


首先，让我们导入需要的库：


In [1]:
import cutlass
import cutlass.cute as cute


### 编写 GPU Kernel
- `@cute.kernel`：这个装饰器告诉 CUTLASS 这个函数应该在 GPU 上运行
- `cute.arch.thread_idx()`：获取当前 GPU thread 的 ID（类似于工人的工号）
- 我们只让一个 thread（thread 0）打印消息，以避免重复打印

In [2]:
@cute.kernel
def kernel():
    # 获取 thread index 的 x 分量（y 和 z 分量未使用）
    tidx, _, _ = cute.arch.thread_idx()
    # 只有第一个 thread（thread 0）打印消息, 否则 32 个线程都会打印
    if tidx == 0:
        cute.printf("Hello world on device")

### 编写 Host 函数

现在我们需要一个函数来设置 GPU 并启动我们的 kernel。
关键概念：
- `@cute.jit`：这个装饰器用于在 CPU 上运行但可以启动 GPU 代码的函数
- 使用 GPU 之前需要初始化 CUDA，`.launch()` 告诉 CUDA 使用多少个 block、thread、shared memory 等

In [3]:
@cute.jit
def hello_world():
    # 从 host 代码打印 hello world
    cute.printf("hello world on host")

    # 启动 kernel
    kernel().launch(
        grid=(1, 1, 1),  # 单个 thread block
        block=(32, 1, 1),  # 每个 thread block 一个 warp（32 个 thread）
    )

### 编译和运行

我们可以通过两种方式来运行程序：

1. 即时编译（JIT）并运行
2. 分离编译，允许我们编译一次代码后多次运行
   
请注意，方法 2 的 `Compiling...` 有可能会在第一个 kernel 的 "Hello world in host" 之前打印。这展示了 CPU 和 GPU 之间的异步行为。

In [ ]:
# 初始化 CUDA context 以启动带有错误检查的 kernel
# 我们显式进行 context 初始化，让用户可以控制 context 的创建, 并避免多个 context 可能带来的问题
# cutlass.cuda.initialize_cuda_context()

# 方法 1：即时编译（JIT）- 立即编译并运行代码
print("Running hello_world()...")
# hello_world()

# 方法 2：先编译（适用于需要多次运行相同代码的情况）
print("Compiling...")
hello_world_compiled = cute.compile(hello_world)

# 编译时导出 PTX/CUBIN 文件
from cutlass.cute import KeepPTX, KeepCUBIN

print("Compiling with PTX/CUBIN dumped...")
# 另外，也可以使用字符串选项进行编译，如：
# cute.compile(hello_world, options="--keep-ptx --keep-cubin")
hello_world_compiled_ptx_on = cute.compile[KeepPTX, KeepCUBIN](hello_world)

# 运行预编译版本
print("Running compiled version...")
hello_world_compiled()

Running hello_world()...
Compiling...
Compiling with PTX/CUBIN dumped...
Running compiled version...
hello world on host


0

Hello world on device


## 二、数据类型



在大多数情况下，CuTe DSL 中的数据结构与 Python 数据结构的工作方式相同，但值得注意的区别是，Python 数据结构在大多数情况下被视为静态数据，由嵌入在 Python 解释器中的 DSL 编译器解释。

### 原始数值类型
为了区分编译时值和运行时值，CuTe DSL 引入了原始类型来表示 JIT 编译代码中的动态值。

CuTe DSL 提供了一套完整的原始数值类型：

| 类型 | 说明 |
|------|------|
| Int8/16/32/64/128 | 有符号整数 |
| Uint8/16/32/64/128 | 无符号整数 |
| Float16/32/64 | 浮点数 |
| BFloat16, TFloat32 | 特殊浮点格式 |
| Float8E4M3, Float8E5M2 | 8位浮点数 |

这些专门类型旨在表示 CuTe DSL 代码中将在运行时求值的动态值，与 Python 内置数值类型（在编译期间求值）形成对比。

使用示例：

```python
x = cutlass.Int32(5)        # 创建一个 32 位整数
y = cutlass.Float32(3.14)   # 创建一个 32 位浮点数

@cute.jit
def foo(a: cutlass.Int32):  # 将 `a` 通过 ABI 传递给 jit 函数的 32 位整数
    ...
```



In [11]:
@cute.jit
def bar():
    a = cutlass.Float32(3.14)
    print("a(static) =", a)  # prints `a(static) = ?`
    cute.printf("a(dynamic) = {}", a)  # prints `a(dynamic) = 3.140000`

    b = cutlass.Int32(5)
    print("b(static) =", b)  # prints `b(static) = 5`
    cute.printf("b(dynamic) = {}", b)  # prints `b(dynamic) = 5`


bar()

a(static) = ?
b(static) = ?
a(dynamic) = 3.140000
b(dynamic) = 5
Hello world on device


### 类型转换 API

CUTLASS 数值类型通过所有 Numeric 类型上可用的 `to()` 方法提供类型转换。这允许你在运行时在不同的数值数据类型之间进行转换。

语法：

```python
new_value = value.to(target_type)
```

`to()` 方法支持以下类型之间的转换：
- 整数类型（Int8、Int16、Int32、Int64、UInt8、UInt16、UInt32、UInt64）
- 浮点类型（Float16、Float32、Float64、BFloat16）
- 混合整数/浮点转换

请注意，从浮点类型转换为整数类型时，小数部分会被截断。在不同范围的类型之间转换时，如果值超出目标类型的可表示范围，可能会被截断或丢失精度。

In [12]:
@cute.jit
def type_conversion():
    # 从 Int32 转换为 Float32
    x = cutlass.Int32(42)
    y = x.to(cutlass.Float32)
    cute.printf("Int32({}) => Float32({})", x, y)

    # 从 Float32 转换为 Int32
    a = cutlass.Float32(3.14)
    b = a.to(cutlass.Int32)
    cute.printf("Float32({}) => Int32({})", a, b)

    # 从 Int32 转换为 Int8
    c = cutlass.Int32(127)
    d = c.to(cutlass.Int8)
    cute.printf("Int32({}) => Int8({})", c, d)

    # 从 Int32 转换为 Int8，值超出 Int8 范围
    e = cutlass.Int32(300)
    f = e.to(cutlass.Int8)
    cute.printf("Int32({}) => Int8({}) (truncated due to range limitation)", e, f)


type_conversion()

Int32(42) => Float32(42.000000)
Float32(3.140000) => Int32(3)
Int32(127) => Int8(127)
Int32(300) => Int8(44) (truncated due to range limitation)


### 运算符重载

CUTLASS 数值类型支持 Python 内置运算符，允许你编写自然的数学表达式。这些运算符既可以与 CUTLASS 数值类型一起使用，也可以与 Python 原生数值类型一起使用。

支持的运算符包括：
- 算术运算：`+`、`-`、`*`、`/`、`//`、`%`、`**`
- 比较运算：`<`、`<=`、`==`、`!=`、`>=`、`>`
- 位运算：`&`、`|`、`^`、`<<`、`>>`
- 一元运算：`-`（取反）、`~`（按位取反）

In [13]:
@cute.jit
def operator_demo():
    # 算术运算符
    a = cutlass.Int32(10)
    b = cutlass.Int32(3)
    cute.printf("a: Int32({}), b: Int32({})", a, b)

    x = cutlass.Float32(5.5)
    cute.printf("x: Float32({})", x)

    cute.printf("")

    sum_result = a + b
    cute.printf("a + b = {}", sum_result)

    y = x * 2  # 与 Python 原生类型相乘
    cute.printf("x * 2 = {}", y)

    # 混合类型算术（Int32 + Float32），整数转换为 float32
    mixed_result = a + x
    cute.printf("a + x = {} (Int32 + Float32 promotes to Float32)", mixed_result)

    # Int32 除法（注意：整数除法）
    div_result = a / b
    cute.printf("a / b = {}", div_result)

    # 浮点除法
    float_div = x / cutlass.Float32(2.0)
    cute.printf("x / 2.0 = {}", float_div)

    # 比较运算符
    is_greater = a > b
    cute.printf("a > b = {}", is_greater)

    # 位运算符
    bit_and = a & b
    cute.printf("a & b = {}", bit_and)

    neg_a = -a
    cute.printf("-a = {}", neg_a)

    not_a = ~a
    cute.printf("~a = {}", not_a)


operator_demo()

a: Int32(10), b: Int32(3)
x: Float32(5.500000)

a + b = 13
x * 2 = 11.000000
a + x = 15.500000 (Int32 + Float32 promotes to Float32)
a / b = 3.333333
x / 2.0 = 2.750000
a > b = 1
a & b = 2
-a = -10
~a = -11


## 三、Tensor



### cute tensor 介绍
CuTe 中的 tensor 通过两个关键组件的组合来创建：

1. **Engine（引擎）**（E）- 一个支持随机访问的类指针对象，支持：
   - 偏移操作：`e + d → e`（按布局值域的元素偏移引擎）
   - 解引用操作：`*e → v`（解引用引擎以产生值）

2. **Layout（布局）**（L）- 定义从坐标到偏移量的映射

Tensor 被正式定义为引擎 E 与布局 L 的组合，表示为 `T = E ∘ L`。在坐标 c 处评估 tensor 时，它：

1. 使用布局将坐标 c 映射到值域
2. 相应地偏移引擎
3. 解引用结果以获得 tensor 的值

这可以用数学形式表示为：

```
T(c) = (E ∘ L)(c) = *(E + L(c))
```


以下是一个简单示例，展示如何使用指针和布局 `(8,5):(5,1)` 创建 tensor 并用 1 填充：

In [14]:
@cute.jit
def create_tensor_from_ptr(ptr: cute.Pointer):
    layout = cute.make_layout((8, 5), stride=(5, 1))
    tensor = cute.make_tensor(ptr, layout)
    tensor.fill(1)
    cute.print_tensor(tensor)

这创建了一个 tensor，其中：
- 引擎是一个指针
- 布局的 shape 为 `(8, 5)`，stride 为 `(5, 1)`
- 生成的 tensor 可以使用布局定义的坐标进行访问

我们可以通过使用 torch 分配缓冲区并使用指向 torch tensor 的指针运行测试来验证这一点

In [15]:
import torch

from cutlass.torch import dtype as torch_dtype
import cutlass.cute.runtime as cute_rt

a = torch.randn(8, 5, dtype=torch_dtype(cutlass.Float32))
ptr_a = cute_rt.make_ptr(cutlass.Float32, a.data_ptr())

create_tensor_from_ptr(ptr_a)

tensor(raw_ptr(0x000000000c76d440: f32, generic, align<4>) o (8,5):(5,1), data=
       [[ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        ...
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ],
        [ 1.000000,  1.000000,  1.000000,  1.000000,  1.000000, ]])


### DLPACK 支持

CuTe DSL 原生支持 dlpack 协议。DLPack 定义了一个通用的、与框架无关的内存布局描述符（DLTensor），让不同框架可以直接访问彼此的张量数据，而无需复制内存。这使得与支持 DLPack 的框架（如 torch、numpy、jax、tensorflow 等）可以轻松集成。

更多信息请参考 DLPACK 项目：https://github.com/dmlc/dlpack


与 Python 标准协议相同，cuteDSL中调用 `from_dlpack` 可以转换任何支持 `__dlpack__` 和 `__dlpack_device__` 的 tensor 或 ndarray 对象。

In [16]:
from cutlass.cute.runtime import from_dlpack


@cute.jit
def print_tensor_dlpack(src: cute.Tensor):
    print(src)
    cute.print_tensor(src)

In [17]:
a = torch.randn(8, 5, dtype=torch_dtype(cutlass.Float32))

print_tensor_dlpack(from_dlpack(a))

tensor<ptr<f32, generic> o (8,5):(5,1)>
tensor(raw_ptr(0x000000000cbb2bc0: f32, generic, align<4>) o (8,5):(5,1), data=
       [[-0.925875,  0.417654, -0.916446, -2.299654,  0.037267, ],
        [ 0.054994,  0.320543, -0.232193,  0.032619, -0.734149, ],
        [ 0.471200,  1.266203,  0.547101,  0.106266, -0.627453, ],
        ...
        [ 0.541711, -0.357838,  0.720978,  0.189100,  1.243145, ],
        [ 1.226349,  0.330756,  0.756583,  0.264087,  0.666108, ],
        [ 0.081440, -0.629994,  0.376176,  0.710784, -0.228333, ]])


In [18]:
import numpy as np

a = np.random.randn(8, 8).astype(np.float32)

# 使用 from_dlpack 直接传递 Pytorch 的 tensor
print_tensor_dlpack(from_dlpack(a))

tensor<ptr<f32, generic> o (8,8):(8,1)>


tensor(raw_ptr(0x000000000c12ea30: f32, generic, align<4>) o (8,8):(8,1), data=
       [[ 0.439202, -0.415553, -0.957999, ...,  0.532141, -0.476766, -0.840787, ],
        [ 1.369587, -0.193902, -0.171274, ..., -0.835015, -0.811520, -0.963730, ],
        [-1.261955, -1.898622,  1.832979, ..., -0.270350,  0.879817, -1.927020, ],
        ...
        [ 0.780666,  0.776979, -0.493811, ..., -0.394934, -0.924691,  0.179890, ],
        [ 0.766864,  0.688052,  0.170053, ..., -2.341738,  0.079710,  0.169724, ],
        [ 0.070873, -0.371389, -0.256903, ...,  1.072019, -1.530429, -0.798877, ]])


### Tensor 求值方法

Tensor 支持两种主要的求值方法：

1. 完全求值:
当使用完整坐标 c 进行 tensor 求值时，它计算偏移量，将其应用于引擎，并解引用以返回存储的值。这是你想要访问 tensor 中特定元素的直接情况。

2. 部分求值（切片）:
当使用不完整坐标 c = c' ⊕ c*（其中 c* 表示未指定的部分）进行求值时，结果是一个新的 tensor，它是原始 tensor 的一个切片，其引擎偏移量考虑了提供的坐标。这个操作可以表示为：

```
T(c) = (E ∘ L)(c) = (E + L(c')) ∘ L(c*) = T'(c*)
```

切片有效地降低了 tensor 的维度，创建一个可以进一步求值或操作的子 tensor。

In [19]:
@cute.jit
def tensor_access_item(a: cute.Tensor):
    # 使用线性索引访问数据
    cute.printf(
        "a[2] = {} (equivalent to a[{}])",
        a[2],
        cute.make_identity_tensor(a.layout.shape)[2],
    )
    cute.printf(
        "a[9] = {} (equivalent to a[{}])",
        a[9],
        cute.make_identity_tensor(a.layout.shape)[9],
    )

    # 使用 n-d 坐标访问数据，以下两种方式等价
    cute.printf("a[2,0] = {}", a[2, 0])
    cute.printf("a[2,4] = {}", a[2, 4])
    cute.printf("a[(2,4)] = {}", a[2, 4])

    # 给 tensor@(2,4) 赋值
    a[2, 3] = 100.0
    a[2, 4] = 101.0
    cute.printf("a[2,3] = {}", a[2, 3])
    cute.printf("a[(2,4)] = {}", a[(2, 4)])


# 使用 torch 创建一个包含顺序数据的 tensor
data = torch.arange(0, 8 * 5, dtype=torch.float32).reshape(8, 5)
tensor_access_item(from_dlpack(data))

print(data)

a[2] = 10.000000 (equivalent to a[(2,0)])
a[9] = 6.000000 (equivalent to a[(1,1)])
a[2,0] = 10.000000
a[2,4] = 14.000000
a[(2,4)] = 14.000000
a[2,3] = 100.000000
a[(2,4)] = 101.000000
tensor([[  0.,   1.,   2.,   3.,   4.],
        [  5.,   6.,   7.,   8.,   9.],
        [ 10.,  11.,  12., 100., 101.],
        [ 15.,  16.,  17.,  18.,  19.],
        [ 20.,  21.,  22.,  23.,  24.],
        [ 25.,  26.,  27.,  28.,  29.],
        [ 30.,  31.,  32.,  33.,  34.],
        [ 35.,  36.,  37.,  38.,  39.]])


### Tensor 作为内存视图

在 CUDA 编程中，不同的内存空间在访问速度、作用域和生命周期方面有不同的特性：

- **generic**：可以引用任何其他内存空间的默认内存空间。
- **global memory（gmem）**：所有 block 的所有 thread 都可访问，但延迟较高。
- **shared memory（smem）**：同一 block 内的所有 thread 都可访问，延迟比 global memory 低得多。
- **register memory（rmem）**：thread 私有内存，延迟最低，但容量有限。
- **tensor memory（tmem）**：NVIDIA Blackwell 架构中引入的专门用于 tensor 操作的内存。

在 CuTe 中创建 tensor 时，你可以根据访问模式指定内存空间来优化性能。

有关 CUDA 内存空间的更多信息，请参阅 [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html#memory-hierarchy)。


### 坐标 Tensor


坐标 tensor $T: Z^n → Z^m$ 是一种数学结构，用于建立坐标空间之间的映射。与将坐标映射到标量值的标准 tensor 不同，坐标 tensor 将坐标映射到其他坐标，形成 tensor 操作和变换的基本构建块。


考虑一个 `(4,4)` 坐标 tensor：

**行优先布局（C 风格）：**
\begin{bmatrix} 
(0,0) & (0,1) & (0,2) & (0,3) \\
(1,0) & (1,1) & (1,2) & (1,3) \\
(2,0) & (2,1) & (2,2) & (2,3) \\
(3,0) & (3,1) & (3,2) & (3,3)
\end{bmatrix}

**列优先布局（Fortran 风格）：**
\begin{bmatrix}
(0,0) & (1,0) & (2,0) & (3,0) \\
(0,1) & (1,1) & (2,1) & (3,1) \\
(0,2) & (1,2) & (2,2) & (3,2) \\
(0,3) & (1,3) & (2,3) & (3,3)
\end{bmatrix}

恒等 tensor $I$ 是实现恒等映射函数的坐标 tensor 的特例：

**定义：**
对于给定的形状 $S = (s_1, s_2, ..., s_n)$，恒等 tensor $I$ 满足：$I(c) = c, \forall c \in \prod_{i=1}^n [0, s_i)$

**属性：**
1. **双射映射**：恒等 tensor 在坐标之间建立一一对应关系。
2. **布局不变性**：无论底层内存布局如何，逻辑结构保持不变。
3. **坐标保持**：对于任意坐标 c，I(c) = c。


CuTe 通过字典序建立 1-D 索引和 N-D 坐标之间的同构。对于形状为 S = (s₁, s₂, ..., sₙ) 的恒等 tensor 中的坐标 c = (c₁, c₂, ..., cₙ)：

**线性索引公式：**
$\text{idx} = c_1 + \sum_{i=2}^{n} \left(c_i \prod_{j=1}^{i-1} s_j\right)$

**示例：**
```python
# 从给定形状创建恒等 tensor
coord_tensor = make_identity_tensor(layout.shape())

# 使用线性索引访问坐标
coord = coord_tensor[linear_idx]  # 返回 N-D 坐标
```

这种双向映射实现了从线性索引到 N 维坐标的高效转换，便于 tensor 操作和内存访问模式。

In [20]:
@cute.jit
def print_tensor_coord(a: cute.Tensor):
    coord_tensor = cute.make_identity_tensor(a.layout.shape)
    print(coord_tensor)
    cute.print_tensor(coord_tensor)


a = torch.randn(8, 4, dtype=torch_dtype(cutlass.Float32))
print_tensor_coord(from_dlpack(a))

tensor<(0,0) o (8,4):(1@0,1@1)>
tensor((0,0) o (8,4):(1@0,1@1), data=
       [[ (0,0),  (0,1),  (0,2),  (0,3), ],
        [ (1,0),  (1,1),  (1,2),  (1,3), ],
        [ (2,0),  (2,1),  (2,2),  (2,3), ],
        ...
        [ (5,0),  (5,1),  (5,2),  (5,3), ],
        [ (6,0),  (6,1),  (6,2),  (6,3), ],
        [ (7,0),  (7,1),  (7,2),  (7,3), ]])


## 四、TensorSSA



In [21]:
import cutlass
import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack

import numpy as np

本教程介绍什么是 `TensorSSA` 以及为什么需要它。

### TensorSSA 概念与使用场景

`TensorSSA` 是 CuTe DSL 中静态单赋值（SSA）形式的 tensor 值，可视为驻留在寄存器中的 tensor。它封装 MLIR tensor 值，通过重载 Python 运算符（`+`、`-`、`*`、`/`、`[]` 等）表达逐元素操作和归约，并转换为优化的向量化指令。它是用户计算逻辑与底层 MLIR IR 之间的桥梁。

`TensorSSA` 主要用于以下场景：

1. 内存加载和存储



In [22]:
@cute.jit
def load_and_store(res: cute.Tensor, a: cute.Tensor, b: cute.Tensor):
    """
    从内存加载数据并将结果存储到内存。

    :param res: 用于存储结果的目标 tensor。
    :param a: 要加载的源 tensor。
    :param b: 要加载的源 tensor。
    """
    a_vec = a.load()
    print(f"a_vec: {a_vec}")  # 打印 `a_vec: vector<12xf32> o (3, 4)`
    b_vec = b.load()
    print(f"b_vec: {b_vec}")  # 打印 `b_vec: vector<12xf32> o (3, 4)`
    res.store(a_vec + b_vec)
    cute.print_tensor(res)


a = np.ones(12).reshape((3, 4)).astype(np.float32)
b = np.ones(12).reshape((3, 4)).astype(np.float32)
c = np.zeros(12).reshape((3, 4)).astype(np.float32)
load_and_store(from_dlpack(c), from_dlpack(a), from_dlpack(b))

a_vec: tensor_value<vector<12xf32> o (3, 4)>
b_vec: tensor_value<vector<12xf32> o (3, 4)>
tensor(raw_ptr(0x000000000c98fe40: f32, generic, align<4>) o (3,4):(4,1), data=
       [[ 2.000000,  2.000000,  2.000000,  2.000000, ],
        [ 2.000000,  2.000000,  2.000000,  2.000000, ],
        [ 2.000000,  2.000000,  2.000000,  2.000000, ]])


2. 寄存器级 Tensor 操作

在编写 kernel 逻辑时，需要对加载到寄存器中的数据执行各种计算、变换、切片等操作。

In [23]:
@cute.jit
def apply_slice(src: cute.Tensor, dst: cute.Tensor, indices: cutlass.Constexpr):
    """
    对 src tensor 应用切片操作并将结果存储到 dst tensor。

    :param src: 要进行切片的源 tensor。
    :param dst: 用于存储结果的目标 tensor。
    :param indices: 用于切片源 tensor 的索引。
    """
    src_vec = src.load()
    dst_vec = src_vec[indices]
    print(f"{src_vec} -> {dst_vec}")
    if cutlass.const_expr(isinstance(dst_vec, cute.TensorSSA)):
        dst.store(dst_vec)
        cute.print_tensor(dst)
    else:
        dst[0] = dst_vec
        cute.print_tensor(dst)


def slice_1():
    src_shape = (4, 2, 3)
    dst_shape = (4, 3)
    indices = (None, 1, None)

    """
    a:
    [[[ 0.  1.  2.]
      [ 3.  4.  5.]]

     [[ 6.  7.  8.]
      [ 9. 10. 11.]]

     [[12. 13. 14.]
      [15. 16. 17.]]

     [[18. 19. 20.]
      [21. 22. 23.]]]
    """
    a = np.arange(np.prod(src_shape)).reshape(*src_shape).astype(np.float32)
    dst = np.random.randn(*dst_shape).astype(np.float32)
    apply_slice(from_dlpack(a), from_dlpack(dst), indices)


slice_1()

tensor_value<vector<24xf32> o (4, 2, 3)> -> tensor_value<vector<12xf32> o (4, 3)>
tensor(raw_ptr(0x000000000d59f8f0: f32, generic, align<4>) o (4,3):(3,1), data=
       [[ 3.000000,  4.000000,  5.000000, ],
        [ 9.000000,  10.000000,  11.000000, ],
        [ 15.000000,  16.000000,  17.000000, ],
        [ 21.000000,  22.000000,  23.000000, ]])


In [24]:
def slice_2():
    src_shape = (4, 2, 3)
    dst_shape = (1,)
    indices = 10
    a = np.arange(np.prod(src_shape)).reshape(*src_shape).astype(np.float32)
    dst = np.random.randn(*dst_shape).astype(np.float32)
    apply_slice(from_dlpack(a), from_dlpack(dst), indices)


slice_2()

tensor_value<vector<24xf32> o (4, 2, 3)> -> ?
tensor(raw_ptr(0x00000000036f4040: f32, generic, align<4>) o (1):(1), data=
       [ 13.000000, ])


### 算术运算

如前所述，有许多 tensor 操作的操作数是 `TensorSSA`，它们都是逐元素操作，下面给出一些示例。

1. 二元运算

对于二元运算，左操作数是 `TensorSSA`，右操作数可以是 `TensorSSA` 或 `Numeric`。当右操作数是 `Numeric` 时，它将被广播为 `TensorSSA`。

In [25]:
@cute.jit
def binary_op_1(res: cute.Tensor, a: cute.Tensor, b: cute.Tensor):
    a_vec = a.load()
    b_vec = b.load()

    add_res = a_vec + b_vec
    cute.print_tensor(add_res)  # 打印 [3.000000, 3.000000, 3.000000]

    sub_res = a_vec - b_vec
    cute.print_tensor(sub_res)  # 打印 [-1.000000, -1.000000, -1.000000]

    mul_res = a_vec * b_vec
    cute.print_tensor(mul_res)  # 打印 [2.000000, 2.000000, 2.000000]

    div_res = a_vec / b_vec
    cute.print_tensor(div_res)  # 打印 [0.500000, 0.500000, 0.500000]

    floor_div_res = a_vec // b_vec
    cute.print_tensor(res)  # 打印 [0.000000, 0.000000, 0.000000]

    mod_res = a_vec % b_vec
    cute.print_tensor(mod_res)  # 打印 [1.000000, 1.000000, 1.000000]


a = np.empty((3,), dtype=np.float32)
a.fill(1.0)
b = np.empty((3,), dtype=np.float32)
b.fill(2.0)
res = np.empty((3,), dtype=np.float32)
binary_op_1(from_dlpack(res), from_dlpack(a), from_dlpack(b))

tensor(raw_ptr(0x00007ffe0566bf20: f32, rmem, align<32>) o (3):(1), data=
       [ 3.000000, ],
       [ 3.000000, ],
       [ 3.000000, ])
tensor(raw_ptr(0x00007ffe0566bf40: f32, rmem, align<32>) o (3):(1), data=
       [-1.000000, ],
       [-1.000000, ],
       [-1.000000, ])
tensor(raw_ptr(0x00007ffe0566bf60: f32, rmem, align<32>) o (3):(1), data=
       [ 2.000000, ],
       [ 2.000000, ],
       [ 2.000000, ])
tensor(raw_ptr(0x00007ffe0566bf80: f32, rmem, align<32>) o (3):(1), data=
       [ 0.500000, ],
       [ 0.500000, ],
       [ 0.500000, ])
tensor(raw_ptr(0x000000000d26b6f0: f32, generic, align<4>) o (3):(1), data=
       [ 0.000000, ],
       [ 0.000000, ],
       [ 0.000000, ])
tensor(raw_ptr(0x00007ffe0566bfa0: f32, rmem, align<32>) o (3):(1), data=
       [ 1.000000, ],
       [ 1.000000, ],
       [ 1.000000, ])


In [26]:
@cute.jit
def binary_op_2(res: cute.Tensor, a: cute.Tensor, c: cutlass.Constexpr):
    a_vec = a.load()

    add_res = a_vec + c
    cute.print_tensor(add_res)  # 打印 [3.000000, 3.000000, 3.000000]

    sub_res = a_vec - c
    cute.print_tensor(sub_res)  # 打印 [-1.000000, -1.000000, -1.000000]

    mul_res = a_vec * c
    cute.print_tensor(mul_res)  # 打印 [2.000000, 2.000000, 2.000000]

    div_res = a_vec / c
    cute.print_tensor(div_res)  # 打印 [0.500000, 0.500000, 0.500000]

    floor_div_res = a_vec // c
    cute.print_tensor(floor_div_res)  # 打印 [0.000000, 0.000000, 0.000000]

    mod_res = a_vec % c
    cute.print_tensor(mod_res)  # 打印 [1.000000, 1.000000, 1.000000]


a = np.empty((3,), dtype=np.float32)
a.fill(1.0)
c = 2.0
res = np.empty((3,), dtype=np.float32)
binary_op_2(from_dlpack(res), from_dlpack(a), c)

tensor(raw_ptr(0x00007ffe0566bf40: f32, rmem, align<32>) o (3):(1), data=
       [ 3.000000, ],
       [ 3.000000, ],
       [ 3.000000, ])
tensor(raw_ptr(0x00007ffe0566bf60: f32, rmem, align<32>) o (3):(1), data=
       [-1.000000, ],
       [-1.000000, ],
       [-1.000000, ])
tensor(raw_ptr(0x00007ffe0566bf80: f32, rmem, align<32>) o (3):(1), data=
       [ 2.000000, ],
       [ 2.000000, ],
       [ 2.000000, ])
tensor(raw_ptr(0x00007ffe0566bfa0: f32, rmem, align<32>) o (3):(1), data=
       [ 0.500000, ],
       [ 0.500000, ],
       [ 0.500000, ])
tensor(raw_ptr(0x00007ffe0566bfc0: f32, rmem, align<32>) o (3):(1), data=
       [ 0.000000, ],
       [ 0.000000, ],
       [ 0.000000, ])
tensor(raw_ptr(0x00007ffe0566bfe0: f32, rmem, align<32>) o (3):(1), data=
       [ 1.000000, ],
       [ 1.000000, ],
       [ 1.000000, ])


In [27]:
@cute.jit
def binary_op_3(res: cute.Tensor, a: cute.Tensor, b: cute.Tensor):
    a_vec = a.load()
    b_vec = b.load()

    gt_res = a_vec > b_vec
    res.store(gt_res)

    """
    ge_res = a_ >= b_   # [False, True, False]
    lt_res = a_ < b_    # [True, False, True]
    le_res = a_ <= b_   # [True, False, True]
    eq_res = a_ == b_   # [False, False, False]
    """


a = np.array([1, 2, 3], dtype=np.float32)
b = np.array([2, 1, 4], dtype=np.float32)
res = np.empty((3,), dtype=np.bool_)
binary_op_3(from_dlpack(res), from_dlpack(a), from_dlpack(b))
print(res)  # 打印 [False, True, False]

[False  True False]


In [28]:
@cute.jit
def binary_op_4(res: cute.Tensor, a: cute.Tensor, b: cute.Tensor):
    a_vec = a.load()
    b_vec = b.load()

    xor_res = a_vec ^ b_vec
    res.store(xor_res)

    # or_res = a_vec | b_vec
    # res.store(or_res)     # 打印 [3, 2, 7]

    # and_res = a_vec & b_vec
    # res.store(and_res)      # 打印 [0, 2, 0]


a = np.array([1, 2, 3], dtype=np.int32)
b = np.array([2, 2, 4], dtype=np.int32)
res = np.empty((3,), dtype=np.int32)
binary_op_4(from_dlpack(res), from_dlpack(a), from_dlpack(b))
print(res)  # 打印 [3, 0, 7]

[3 0 7]


2. 一元运算

In [29]:
@cute.jit
def unary_op_1(res: cute.Tensor, a: cute.Tensor):
    a_vec = a.load()

    sqrt_res = cute.math.sqrt(a_vec)
    cute.print_tensor(sqrt_res)  # 打印 [2.000000, 2.000000, 2.000000]

    sin_res = cute.math.sin(a_vec)
    res.store(sin_res)
    cute.print_tensor(sin_res)  # 打印 [-0.756802, -0.756802, -0.756802]

    exp2_res = cute.math.exp2(a_vec)
    cute.print_tensor(exp2_res)  # 打印 [16.000000, 16.000000, 16.000000]


a = np.array([4.0, 4.0, 4.0], dtype=np.float32)
res = np.empty((3,), dtype=np.float32)
unary_op_1(from_dlpack(res), from_dlpack(a))

tensor(raw_ptr(0x00007ffe0566bf80: f32, rmem, align<32>) o (3):(1), data=
       [ 2.000000, ],
       [ 2.000000, ],
       [ 2.000000, ])
tensor(raw_ptr(0x00007ffe0566bfa0: f32, rmem, align<32>) o (3):(1), data=
       [-0.756802, ],
       [-0.756802, ],
       [-0.756802, ])
tensor(raw_ptr(0x00007ffe0566bfc0: f32, rmem, align<32>) o (3):(1), data=
       [ 16.000000, ],
       [ 16.000000, ],
       [ 16.000000, ])


3. 归约运算

`TensorSSA` 的 `reduce` 方法从一个初始值开始，应用指定的归约操作（`ReductionOp.ADD`、
`ReductionOp.MUL`、`ReductionOp.MAX`、`ReductionOp.MIN`），并沿着 `reduction_profile` 
指定的维度执行归约。结果通常是一个维度减少的新 `TensorSSA`，如果对所有轴进行归约则返回标量值。

In [30]:
@cute.jit
def reduction_op(a: cute.Tensor):
    """
    对 src tensor 应用归约操作。

    :param src: 要进行归约的源 tensor。
    """
    a_vec = a.load()
    red_res = a_vec.reduce(cute.ReductionOp.ADD, 0.0, reduction_profile=0)
    cute.printf(red_res)  # 打印 21.000000

    red_res = a_vec.reduce(cute.ReductionOp.ADD, 0.0, reduction_profile=(None, 1))
    cute.print_tensor(red_res)  # 打印 [6.000000, 15.000000]

    red_res = a_vec.reduce(cute.ReductionOp.ADD, 1.0, reduction_profile=(1, None))
    cute.print_tensor(red_res)  # 打印 [6.000000, 8.000000, 10.000000]


a = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
reduction_op(from_dlpack(a))

21.000000
tensor(raw_ptr(0x00007ffe0566bf60: f32, rmem, align<32>) o (2):(1), data=
       [ 6.000000, ],
       [ 15.000000, ])
tensor(raw_ptr(0x00007ffe0566bf80: f32, rmem, align<32>) o (3):(1), data=
       [ 6.000000, ],
       [ 8.000000, ],
       [ 10.000000, ])


4. 广播

`TensorSSA` 支持遵循 NumPy 广播规则的广播操作。当满足特定条件时，广播允许你对不同形状的数组执行操作。关键规则是：

1. 源形状用 1 填充以匹配目标形状的秩
2. 源形状每个模式的大小必须为 1 或等于目标形状
3. 广播后，所有模式应匹配目标形状

让我们看一些广播操作的示例：

In [31]:
import cutlass
import cutlass.cute as cute


@cute.jit
def broadcast_examples():
    a = cute.make_rmem_tensor((1, 3), dtype=cutlass.Float32)
    a[0] = 0.0
    a[1] = 1.0
    a[2] = 2.0
    a_val = a.load()
    cute.print_tensor(a_val.broadcast_to((4, 3)))
    # tensor(raw_ptr(0x00007ffe26625740: f32, rmem, align<32>) o (4,3):(1,4), data=
    #    [[ 0.000000,  1.000000,  2.000000, ],
    #     [ 0.000000,  1.000000,  2.000000, ],
    #     [ 0.000000,  1.000000,  2.000000, ],
    #     [ 0.000000,  1.000000,  2.000000, ]])

    c = cute.make_rmem_tensor((4, 1), dtype=cutlass.Float32)
    c[0] = 0.0
    c[1] = 1.0
    c[2] = 2.0
    c[3] = 3.0
    cute.print_tensor(a.load() + c.load())
    # tensor(raw_ptr(0x00007ffe26625780: f32, rmem, align<32>) o (4,3):(1,4), data=
    #        [[ 0.000000,  1.000000,  2.000000, ],
    #         [ 1.000000,  2.000000,  3.000000, ],
    #         [ 2.000000,  3.000000,  4.000000, ],
    #         [ 3.000000,  4.000000,  5.000000, ]])


broadcast_examples()

tensor(raw_ptr(0x00007ffe0566bfa0: f32, rmem, align<32>) o (4,3):(1,4), data=
       [[ 0.000000,  1.000000,  2.000000, ],
        [ 0.000000,  1.000000,  2.000000, ],
        [ 0.000000,  1.000000,  2.000000, ],
        [ 0.000000,  1.000000,  2.000000, ]])
tensor(raw_ptr(0x00007ffe0566bfe0: f32, rmem, align<32>) o (4,3):(1,4), data=
       [[ 0.000000,  1.000000,  2.000000, ],
        [ 1.000000,  2.000000,  3.000000, ],
        [ 2.000000,  3.000000,  4.000000, ],
        [ 3.000000,  4.000000,  5.000000, ]])


上面的示例演示了两个关键的广播场景：

1. **行向量广播**：形状 (1, 3) 的行向量广播到 (4, 3) 时，值沿第一维重复。
2. **列向量+行向量相加**：形状 (1, 3) 与 (4, 1) 相加时，两者自动广播到 (4, 3)，遵循 NumPy 广播规则（维度为 1 或匹配目标大小）。



## 五、打印调试



本 notebook 演示了在 CuTe 中打印值的不同方式，并解释了静态（编译时）和动态（运行时）值之间的重要区别。

In [32]:
import cutlass
import cutlass.cute as cute
import numpy as np

### Print 示例函数

我们来演示 Cute Print 的几个重要概念：

1. Python 的 `print` 与 CuTe 的 `cute.printf`
- `print`：只能在编译时显示静态值
- `cute.printf`：可以在运行时显示静态和动态值

2. 值类型
- `a`：动态 `Int32` 值（运行时）
- `b`：静态 `Constexpr[int]` 值（编译时）

3. 布局打印
展示在静态与动态上下文中布局的不同表示方式：
- 静态上下文：未知值显示为 `?`
- 动态上下文：显示实际值

In [33]:
@cute.jit
def print_example(a: cutlass.Int32, b: cutlass.Constexpr[int]):
    """
    演示 CuTe 中不同的打印方法以及它们如何处理静态与动态值。

    本示例展示：
    1. Python 的 `print` 函数如何在编译时处理静态值，但无法显示动态值
    2. `cute.printf` 如何在运行时显示静态和动态值
    3. 静态与动态上下文中类型的差异
    4. 两种打印方法中布局的表示方式

    Args:
        a: 将在运行时确定的动态 Int32 值
        b: 静态（编译时常量）整数值
    """
    # 使用 Python `print` 打印静态信息
    print(">>>", b)  # => 2
    # `a` 是动态值
    print(">>>", a)  # => ?

    # 使用 `cute.printf` 打印动态信息
    cute.printf(">?? {}", a)  # => 8
    cute.printf(">?? {}", b)  # => 2

    print(">>>", type(a))  # => <class 'cutlass.Int32'>
    print(">>>", type(b))  # => <class 'int'>

    layout = cute.make_layout((a, b))
    print(">>>", layout)  # => (?,2):(1,?)
    cute.printf(">?? {}", layout)  # => (8,2):(1,8)

### 编译和运行

**直接编译并运行**
  - `print_example(cutlass.Int32(8), 2)`
  - 一步编译并运行将同时执行静态和动态打印
    * `>>>` 表示静态打印
    * `>??` 表示动态打印

In [34]:
print_example(cutlass.Int32(8), 2)

>>> 2
>>> ?
>>> Int32
>>> <class 'int'>
>>> (?,2):(1,?)
>?? 8
>?? 2
>?? (8,2):(1,8)


编译函数

当使用 `cute.compile(print_example, cutlass.Int32(8), 2)` 编译函数时，Python 解释器会追踪代码，只评估静态表达式并打印静态信息。

In [35]:
print_example_compiled = cute.compile(print_example, cutlass.Int32(8), 2)

>>> 2
>>> ?
>>> Int32
>>> <class 'int'>
>>> (?,2):(1,?)


调用已编译函数

只打印运行时信息

In [36]:
print_example_compiled(cutlass.Int32(8))

>?? 8
>?? 2
>?? (8,2):(1,8)


0

### 格式字符串示例

`format_string_example` 函数展示了一个重要的限制：
- CuTe 中的 f-string 在编译时求值
- 这意味着动态值不会在 f-string 中显示它们的运行时值
- 当你需要查看运行时值时使用 `cute.printf`

In [37]:
@cute.jit
def format_string_example(a: cutlass.Int32, b: cutlass.Constexpr[int]):
    """
    格式字符串在编译时求值。
    """
    print(f"a: {a}, b: {b}")

    layout = cute.make_layout((a, b))
    print(f"layout: {layout}")


print("Direct run output:")
format_string_example(cutlass.Int32(8), 2)

Direct run output:
a: ?, b: 2
layout: (?,2):(1,?)


### 打印 Tensor 示例

CuTe 通过 `print_tensor` 操作提供了专门的 tensor 打印功能。`cute.print_tensor` 接受以下参数：
- `Tensor`（必需）：你想要打印的 CuTe tensor 对象。该 tensor 必须支持 load 和 store 操作
- `verbose`（可选，默认=False）：一个布尔标志，控制输出的详细程度。设置为 True 时，将打印 tensor 中每个元素的索引详情。

下面的示例代码展示了 verbose 开启和关闭的区别，以及如何打印给定 tensor 的子范围。

In [38]:
from cutlass.cute.runtime import from_dlpack


@cute.jit
def print_tensor_basic(x: cute.Tensor):
    # 打印 tensor
    print("Basic output:")
    cute.print_tensor(x)


@cute.jit
def print_tensor_verbose(x: cute.Tensor):
    # 以详细模式打印 tensor
    print("Verbose output:")
    cute.print_tensor(x, verbose=True)


@cute.jit
def print_tensor_slice(x: cute.Tensor, coord: tuple):
    # 从 3D tensor 中切片出一个 2D tensor
    sliced_data = cute.slice_(x, coord)
    y = cute.make_rmem_tensor(sliced_data.layout, sliced_data.element_type)
    # 通过将切片数据加载到 fragment 中，转换为 TensorSSA 格式
    y.store(sliced_data.load())
    print("Slice output:")
    cute.print_tensor(y)

默认的 `cute.print_tensor` 将输出包含数据类型、存储空间、CuTe 布局信息的 CuTe tensor，并以 torch 风格的格式打印数据。

In [39]:
def tensor_print_example1():
    shape = (4, 3, 2)

    # 创建 [0,...,23] 并 reshape 为 (4, 3, 2)
    data = np.arange(24, dtype=np.float32).reshape(*shape)

    print_tensor_basic(from_dlpack(data))


tensor_print_example1()

Basic output:
tensor(raw_ptr(0x000000000c9f3310: f32, generic, align<4>) o (4,3,2):(6,2,1), data=
       [[[ 0.000000,  2.000000,  4.000000, ],
         [ 6.000000,  8.000000,  10.000000, ],
         [ 12.000000,  14.000000,  16.000000, ],
         [ 18.000000,  20.000000,  22.000000, ]],

        [[ 1.000000,  3.000000,  5.000000, ],
         [ 7.000000,  9.000000,  11.000000, ],
         [ 13.000000,  15.000000,  17.000000, ],
         [ 19.000000,  21.000000,  23.000000, ]]])


详细打印将显示 tensor 中每个元素的坐标详情。下面的示例展示了如何在 2D 4x3 tensor 空间中索引元素。

In [40]:
def tensor_print_example2():
    shape = (4, 3)

    # 创建 [0,...,11] 并 reshape 为 (4, 3)
    data = np.arange(12, dtype=np.float32).reshape(*shape)

    print_tensor_verbose(from_dlpack(data))


tensor_print_example2()

Verbose output:
tensor(raw_ptr(0x000000000d59f8f0: f32, generic, align<4>) o (4,3):(3,1), data= (
	(0,0)= 0.000000
	(1,0)= 3.000000
	(2,0)= 6.000000
	(3,0)= 9.000000
	(0,1)= 1.000000
	(1,1)= 4.000000
	(2,1)= 7.000000
	(3,1)= 10.000000
	(0,2)= 2.000000
	(1,2)= 5.000000
	(2,2)= 8.000000
	(3,2)= 11.000000
)


要打印给定 Tensor 中的元素子集，我们可以使用 cute.slice_ 选择给定 tensor 的一个范围，将它们加载到寄存器中，然后使用 `cute.print_tensor` 打印值。

In [41]:
def tensor_print_example3():
    shape = (4, 3)

    # 创建 [0,...,11] 并 reshape 为 (4, 3)
    data = np.arange(12, dtype=np.float32).reshape(*shape)

    print_tensor_slice(from_dlpack(data), (None, 0))
    print_tensor_slice(from_dlpack(data), (1, None))


tensor_print_example3()

Slice output:


tensor(raw_ptr(0x00007ffe0566c000: f32, rmem, align<32>) o (4):(3), data=
       [ 0.000000, ],
       [ 3.000000, ],
       [ 6.000000, ],
       [ 9.000000, ])
Slice output:
tensor(raw_ptr(0x00007ffe0566c000: f32, rmem, align<32>) o (3):(1), data=
       [ 3.000000, ],
       [ 4.000000, ],
       [ 5.000000, ])


要打印设备内存中的 tensor，你可以在 CuTe JIT kernel 中使用 `cute.print_tensor`。

In [42]:
@cute.kernel
def print_tensor_gpu(src: cute.Tensor):
    print(src)
    cute.print_tensor(src)


@cute.jit
def print_tensor_host(src: cute.Tensor):
    print_tensor_gpu(src).launch(grid=(1, 1, 1), block=(1, 1, 1))

In [43]:
import torch


def tensor_print_example4():
    a = torch.randn(4, 3, device="cuda")
    cutlass.cuda.initialize_cuda_context()
    print_tensor_host(from_dlpack(a))


tensor_print_example4()

tensor<ptr<f32, gmem> o (4,3):(3,1)>


目前，`cute.print_tensor` 只支持整数数据类型和 `Float16`/`Float32`/`Float64` 浮点数据类型的 tensor。我们将在未来支持更多数据类型。

## 六、CUDA Graphs


### CUDA Graphs 概述

本示例演示如何通过 PyTorch 与 CuTe DSL 一起使用 CUDA graph。
与 PyTorch 的 CUDA graph 实现交互需要将 PyTorch 的 CUDA stream 暴露给 CUTLASS。

要在 Blackwell 上使用 CUDA graph，需要支持 Blackwell 的 PyTorch 版本。
可以通过以下方式获取：
- [PyTorch NGC](https://catalog.ngc.nvidia.com/orgs/nvidia/containers/pytorch)
- [PyTorch 2.7 配合 CUDA 12.8 或更高版本](https://pytorch.org/)（例如，`pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128`）
- 使用你的 CUDA 版本直接构建 PyTorch。

In [44]:
# 导入 torch 用于 CUDA graph
import torch
import cutlass.cute as cute

# 从 cuda driver bindings 导入 CUstream 类型
from cuda.bindings.driver import CUstream

# 从 torch 导入 current_stream 函数
from torch.cuda import current_stream

### Kernel 创建

我们创建一个打印 "Hello world" 的 kernel，以及一个启动 kernel 的 host 函数。
然后通过传入默认 stream 来编译 kernel，以便在我们的 graph 中使用。

在 graph 捕获之前编译 kernel 是必需的，因为 CUDA graph 无法在 graph 执行期间 JIT 编译 kernel。

In [45]:
@cute.kernel
def hello_world_kernel():
    """
    一个打印 hello world 的 kernel
    """
    cute.printf("Hello world")


@cute.jit
def hello_world(stream: CUstream):
    """
    在 stream 中启动 (1,1,1), (1,1,1) grid 的 host 函数
    """
    hello_world_kernel().launch(grid=[1, 1, 1], block=[1, 1, 1], stream=stream)


# 从 PyTorch 获取 stream，这也会初始化我们的 context
# 因此可以省略 cutlass.cuda.initialize_cuda_context()
stream = current_stream()
hello_world_compiled = cute.compile(hello_world, CUstream(stream.cuda_stream))

tensor(raw_ptr(0x00007f3519a00000: f32, gmem, align<4>) o (4,3):(3,1), data=
       [[ 1.039039,  0.352937, -0.386477, ],
        [-0.945468, -0.130710, -1.063859, ],
        [-0.373478,  1.039198, -0.702384, ],
        [-1.403910, -0.338552, -2.292463, ]])


### 创建和重放 CUDA Graph

我们通过 torch 创建一个 stream 和一个 graph。
创建 graph 时，可以将要捕获的 stream 传递给 torch。类似地，我们将 stream 作为 CUstream 传递来运行编译后的 kernel。

最后我们可以重放 graph 并同步。

In [46]:
# 创建一个 CUDA Graph
g = torch.cuda.CUDAGraph()
# 捕获我们的 graph
with torch.cuda.graph(g):
    # 将 torch Stream 转换为 cuStream stream
    # 通过 .cuda_stream 获取底层的 CUstream 来实现
    graph_stream = CUstream(current_stream().cuda_stream)
    # 运行编译后的 kernel 2 次迭代
    for _ in range(2):
        # 在 stream 中运行我们的 kernel
        hello_world_compiled(graph_stream)

# 重放我们的 graph
g.replay()
# 同步所有 stream（相当于 C++ 中的 cudaDeviceSynchronize()）
torch.cuda.synchronize()

Hello world
Hello world


在 NSight Systems 中查看时，我们的运行结果如下：

![两个 hello world kernel 在 CUDA graph 中连续运行的图像](images/cuda_graphs_image.png)

我们可以观察到两个 kernel 的启动，后面跟着一个 `cudaDeviceSynchronize()`。

现在我们可以确认这最小化了一些启动开销：

In [47]:
# 从 PyTorch 获取 CUDA stream
stream = CUstream(current_stream().cuda_stream)

# 创建一个包含 100 次迭代的更大 CUDA Graph
g = torch.cuda.CUDAGraph()
# 捕获我们的 graph
with torch.cuda.graph(g):
    # 将 torch Stream 转换为 cuStream stream
    # 通过 .cuda_stream 获取底层的 CUstream 来实现
    graph_stream = CUstream(current_stream().cuda_stream)
    # 运行编译后的 kernel 100 次迭代
    for _ in range(100):
        # 在 stream 中运行我们的 kernel
        hello_world_compiled(graph_stream)

# 创建用于测量性能的 CUDA event
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

# 运行 kernel 进行 GPU 预热
for _ in range(100):
    hello_world_compiled(stream)

# 记录开始时间
start.record()
# 运行 100 个 kernel
for _ in range(100):
    hello_world_compiled(stream)
# 记录结束时间
end.record()
# 同步 (cudaDeviceSynchronize())
torch.cuda.synchronize()

# 计算在 stream 中启动 kernel 所花费的时间
# 结果单位为 ms
stream_time = start.elapsed_time(end)

# 再次预热 GPU
g.replay()
# 记录开始时间
start.record()
# 运行我们的 graph
g.replay()
# 记录结束时间
end.record()
# 同步 (cudaDeviceSynchronize())
torch.cuda.synchronize()

# 计算在 graph 中启动 kernel 所花费的时间
# 单位为 ms
graph_time = start.elapsed_time(end)

Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hello world
Hell

In [48]:
# 打印使用 CUDA graph 时的加速比
percent_speedup = (stream_time - graph_time) / graph_time
print(f"{percent_speedup * 100.0:.2f}% speedup when using CUDA graphs for this kernel!")

8.69% speedup when using CUDA graphs for this kernel!
